In [4]:
import os
import cv2
from natsort import natsorted
import glob
import ast

def list_files_with_extension(dir_path, extension):
    '''
    列出指定目录下具有特定后缀的文件。
    
    :param dir_path: 要搜索的目录路径
    :param extension: 文件的后缀名（例如 '.txt'）
    :return: 一个包含符合条件的文件路径列表
    '''
    # 使用 glob 模块匹配通配符路径
    search_path = os.path.join(dir_path, f'*{extension}')
    files = natsorted(glob.glob(search_path))
    return files

def uniform_sample(lst, n):
    # 计算需要跳过的步长
    step = len(lst) / float(n)
    sampled_list = []
    # 使用步长进行均匀采样
    for i in range(n):
        index = int(i * step)
        sampled_list.append(lst[index])
    return sampled_list

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="qwen-turbo",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

def acc_(output,label):
    right_1=0
    right_2=0
    right_3=0

    for idx,(i,j) in enumerate(zip(output,label)):
        if idx<10:
            if (i in [True,'yes'] and j in [True,'yes']) or (i in [False,'no'] and j in [False,'no']):
                right_1+=1
        
        if idx>=10 and idx<20:
            if int(i)==int(j):
                right_2+=1

        if idx>=20:
             if llm(i,j)==1:
                right_3+=1

    return right_1,right_2,right_3



Question_dict={
    'office0':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a cup in this scene?",'no'],
         ["Is there a carpet in this scene?",'yes'],
         ["Is there a phone in this scene?",'yes'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'yes'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many sofa are there in this scene?', 4],
         ['How many carpet are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 1],
         ['How many bag are there in this scene?', 1],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'blackboard'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'blackboard'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the sofa?', 'sofa'],
         ['What is the closest object from the table?', 'bag'],
         ['What is the farthest object from the blackboard?', 'door'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the bag?', 'table']],
    
    'office1':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'yes'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'yes'],
         ["Is there a pen in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 2],
         ['How many watch are there in this scene?', 1],
         ['How many towel are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'pillow'],
         ['What is the closest object from the pillow?', 'pillow'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the screen?', 'table'],
         ['What is the closest object from the table?', 'screen'],
         ['What is the farthest object from the blackboard?', 'screen'],
         ['What is the farthest object from the watch?', 'screen'],
         ['What is the closest object from the pen?', 'blackboard']],

    'office2':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 5],
         ['How many table are there in this scene?', 3],
         ['How many watch are there in this scene?', 0],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'sofa'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the sofa?', 'chair'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office3':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 9],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 2],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'watch'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'trash can'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office4':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'no'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 0],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 12],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the farthest object from the door?', 'chair'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the watch?', 'door'],
         ['What is the closest object from the screen?', 'table']],

    'room0':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 8],
         ['How many window are there in this scene?', 3],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 0],
         ['How many sofa are there in this scene?', 4],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'lamp'],
         ['What is the closest object from the sofa?', 'pillow'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'sofa']],

    
    'room1':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a bed in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 0],
         ['How many bed are there in this scene?', 1],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 3],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the bed?', 'pillow'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the cabinet?', 'lamp'],
         ['What is the closest object from the pillow?', 'bed'],
         ['What is the closest object from the picture?', 'bed'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'cabinet']],
    

    'room2':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a bed in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'no'],
         ["Is there a shelf in this scene?",'yes'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many shelf are there in this scene?', 1],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 8],
         ['How many bed are there in this scene?', 0],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 0],
         ['How many lamp are there in this scene?', 0],
         ['How many vase are there in this scene?', 1],

         ['What is the closest object from the door?', 'shelf'],
         ['What is the farthest object from the shelf?', 'picture'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the shelf?', 'vase'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the picture?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the picture?', 'door'],
         ['What is the farthest object from the table?', 'door']],
       
}

In [5]:
import pickle
import base64

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [11]:
R_1,R_2,R_3=[],[],[]

In [12]:
for i in list(Question_dict.keys()):
    jpg_list=list_files_with_extension('/data/coding/eval_replica/Replica/{}/results'.format(i),'jpg')

    base64_image_list=[]
    for j in jpg_list:
        base64_image_list.append(f"data:image/jpg;base64,{encode_image(j)}")

    if len(base64_image_list)>80:
        base64_image_list=uniform_sample(base64_image_list,40)

    output_list=[]

    for iidx,j in enumerate(Question_dict[i]):   ### 遍历每个问题

        if iidx<10:
            prompt="Please answer the question:{} based on the video content,Carefully consider each question,please provide the results directly.please answer yes or no,Do not output punctuation marks and use all lowercase characters".format(j[0])
        if iidx>=10 and iidx<20:
            prompt="Please answer the question:{} based on the video content,Carefully consider each question,please provide the results directly.Only provide Arabic numerals".format(j[0])
        if iidx>=20:
            prompt="Please answer the question:{} based on the video content,Carefully consider each question,please provide the results directly.please answer one words,Do not output punctuation marks and use all lowercase characters".format(j[0])

        completion = client.chat.completions.create(
            model="qwen-vl-max-latest",
            messages=[{"role": "user","content": [
                {"type": "video","video":base64_image_list},
                {"type": "text","text": prompt},
            ]}]
        )
        print(completion.choices[0].message.content)
        output_list.append(completion.choices[0].message.content)

    print(output_list)
    output=output_list
    #output=ast.literal_eval(output)

    r1,r2,r3=acc_(output,[q[1] for q in Question_dict[i]])

    R_1.append(r1)
    R_2.append(r2)
    R_3.append(r3)

yes
yes
yes
yes
yes
no
yes
yes
no
yes
3
1
2
2
1
1
0
0
1
1
trash can
sofa
table
sofa
door
table
chair
door
door
chair
['yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'yes', '3', '1', '2', '2', '1', '1', '0', '0', '1', '1', 'trash can', 'sofa', 'table', 'sofa', 'door', 'table', 'chair', 'door', 'door', 'chair']
yes
yes
yes
yes
no
yes
yes
yes
no
no
2
2
2
1
0
2
0
0
0
0
tissues
pillow
blanket
pillow
tissue box
pillow
monitor
pillow
door
tissue
['yes', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', '2', '2', '2', '1', '0', '2', '0', '0', '0', '0', 'tissues', 'pillow', 'blanket', 'pillow', 'tissue box', 'pillow', 'monitor', 'pillow', 'door', 'tissue']
no
yes
yes
yes
no
yes
yes
no
no
no
0
2
8
1
6
3
0
0
0
0
table
couch
table
couch
door
table
couch
door
door
table
['no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'no', '0', '2', '8', '1', '6', '3', '0', '0', '0', '0', 'table', 'couch', 'table', 'couch', 'door', 'table', 'couch', 'door', 'door', 'table']
yes
y

In [13]:
print(sum(R_1)/80)
print(sum(R_2)/80)
print(sum(R_3)/80)

print(sum(R_1+R_2+R_3)/240)

0.8875
0.5
0.3875
0.5916666666666667


## GLM-4V

In [71]:
import os
import cv2
from natsort import natsorted
import glob
import ast

def list_files_with_extension(dir_path, extension):
    '''
    列出指定目录下具有特定后缀的文件。
    
    :param dir_path: 要搜索的目录路径
    :param extension: 文件的后缀名（例如 '.txt'）
    :return: 一个包含符合条件的文件路径列表
    '''
    # 使用 glob 模块匹配通配符路径
    search_path = os.path.join(dir_path, f'*{extension}')
    files = natsorted(glob.glob(search_path))
    return files

def uniform_sample(lst, n):
    # 计算需要跳过的步长
    step = len(lst) / float(n)
    sampled_list = []
    # 使用步长进行均匀采样
    for i in range(n):
        index = int(i * step)
        sampled_list.append(lst[index])
    return sampled_list

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="qwen-turbo",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

def acc_(output,label):
    right_1=0
    right_2=0
    right_3=0

    for idx,(i,j) in enumerate(zip(output,label)):
        if idx<10:
            if (i in [True,'yes'] and j in [True,'yes']) or (i in [False,'no'] and j in [False,'no']):
                right_1+=1
        
        if idx>=10 and idx<20:
            if int(i)==int(j):
                right_2+=1

        if idx>=20:
             if llm(i,j)==1:
                right_3+=1

    return right_1,right_2,right_3


In [89]:
import base64
from zhipuai import ZhipuAI
client = ZhipuAI(api_key="")

R_1,R_2,R_3=[],[],[]

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="GLM-4-AirX",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

In [91]:
for i in list(question.keys()):
    jpg_list=list_files_with_extension('/data/coding/eval/scene/{}/data'.format(i),'jpg')

    frame = cv2.imread(jpg_list[0])
    height, width, layers = frame.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    # 创建VideoWriter对象，指定输出文件名、编解码器，帧速率和尺寸
    video = cv2.VideoWriter('/data/coding/eval/scene/{}/video.mp4'.format(i), fourcc, 10, (width, height))

    for image in jpg_list:
        video.write(cv2.imread(image))

    cv2.destroyAllWindows()
    video.release()

    video_path='/data/coding/eval/scene/{}/video.mp4'.format(i)
    with open(video_path, 'rb') as video_file:
        video_base = base64.b64encode(video_file.read()).decode('utf-8')

    prompt="Please answer all the questions in the question list:{} based on the video content and output them in the form of ['answer1', 'answer2'...],Carefully consider each question,please provide the results directly".format([q[0] for q in question[i]])

    response = client.chat.completions.create(
        model="glm-4v-plus-0111",  # 填写需要调用的模型名称
        messages=[
        {
            "role": "user",
            "content": [
            {
                "type": "video_url",
                "video_url": {
                    "url" : video_base
                }
            },
            {
                "type": "text",
                "text": prompt
            }
            ]
        }
        ]
    )
    print(response.choices[0].message.content)

    output=completion.choices[0].message.content
    output=ast.literal_eval(output)

    r1,r2,r3=acc_(output,[q[1] for q in question[i]])

    R_1.append(r1)
    R_2.append(r2)
    R_3.append(r3)
    

['yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', '2', '1', '1', '1', '1', '2', '2', '1', '1', '2', 'lamp', 'lamp', 'tv stand', 'tv', 'lamp', 'lamp', 'lamp', 'lamp', 'lamp', 'lamp']
['no', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', '0', '0', '2', '0', '0', 'yes', 'yes', 'yes', 'yes', 'yes', 'curtain', 'cabinet', 'lamp', 'cabinet', 'lamp', 'lamp', 'lamp', 'curtain', 'lamp', 'lamp']
['no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'no', '2', '0', '1', '2', '0', '0', '0', '2', '2', '1', 'cabinet', 'tv', 'cabinet', 'tv', 'cabinet', 'tv', 'tv', 'tv', 'tv', 'tv']
['yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', '2', '2', '1', '2', '2', '3', '1', '2', '1', '1', 'curtain', 'picture', 'blanket', 'cabinet', 'shelf', 'commode', 'tv', 'tv stand', 'ceiling', 'floor']
['no', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', '0', '1', '2', '2', '0', '2', '1', '1', '0', '2', 'tv', 'tv stand', 'wall', 'tv', 'tv', 'tv', 'lamp', 'tv', 'floor', 

KeyboardInterrupt: 

In [88]:
print(sum(R_1)/80)
print(sum(R_2)/80)
print(sum(R_3)/80)

print(sum(R_1+R_2+R_3)/240)

1.025
0.15
0.1
0.425


In [20]:
cv2.imread(jpg_list[0]).shape

(540, 960, 3)

In [2]:
question[]

{'6e67e550-1209-2cd0-8294-7cc2564cf82c': [['Is there a picture in this scene?',
   'yes'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a ceiling in this scene?', 'yes'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a floor in this scene?', 'yes'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a car in this scene?', 'no'],
  ['How many wall are there in this scene?', 3],
  ['How many door are there in this scene?', 1],
  ['How many commode are there in this scene?', 1],
  ['How many curtain are there in this scene?', 1],
  ['How many floor are there in this scene?', 1],
  ['How many shelf are there in this scene?', 1],
  ['How many pillow are there in this scene?', 3],
  ['How many blanket are there in this scene?', 1],
  ['How many lamp are there in this scene?', 1],
  ['How many picture are there in this scene?', 1],
  ['What

In [1]:
import json

In [3]:
data=json.load(open('/data/coding/eval/scene/6e67e550-1209-2cd0-8294-7cc2564cf82c/semseg.v2.json','r'))

label_list=[]
for i in data['segGroups']:
    label_list.append(i['label'])

In [4]:
label_list

['cabinet',
 'cabinet',
 'wall',
 'ceiling',
 'cabinet',
 'floor',
 'cabinet',
 'tv stand',
 'shelf',
 'wall',
 'curtain',
 'tv',
 'picture',
 'lamp',
 'door',
 'wall',
 'commode',
 'pillow',
 'pillow',
 'pillow',
 'blanket']

In [1]:
from PIL import Image
import numpy as np
import io
import base64

# 假设你已经有一个 numpy 数组形式的 img
# img = np.array(...) 

# 如果你是从文件读取图像到 numpy 数组，可以这样做：
img = np.array(Image.open('/data/coding/eval/scene/6e67e550-1209-2cd0-8294-7cc2564cf82c/data/frame-000000.color.jpg'))

# 确保图像数据是有效的，并且被正确地转换回Image对象
# 因为我们将使用PIL的Image对象来保存图像到内存中
img_pil = Image.fromarray(img)

# 创建一个字节流管道，用于存储图像数据而不写入磁盘
buffered = io.BytesIO()

# 将numpy数组转换成的Image对象保存为PNG格式到字节流中
img_pil.save(buffered, format="PNG")

# 将字节流转成base64编码
img_str = base64.b64encode(buffered.getvalue())

# 如果需要字符串形式，而不是字节形式，则解码
img_base64 = img_str.decode('utf-8')

In [2]:
img

array([[[ 30,  10,   3],
        [ 30,  10,   3],
        [ 31,  11,   4],
        ...,
        [ 63,  41,  18],
        [ 67,  41,  16],
        [ 70,  44,  17]],

       [[ 31,  11,   4],
        [ 31,  11,   4],
        [ 31,  11,   4],
        ...,
        [ 63,  41,  18],
        [ 66,  42,  16],
        [ 71,  45,  18]],

       [[ 31,  12,   5],
        [ 30,  11,   4],
        [ 30,  11,   4],
        ...,
        [ 62,  40,  17],
        [ 67,  43,  17],
        [ 72,  48,  20]],

       ...,

       [[160, 152, 133],
        [161, 153, 134],
        [160, 152, 133],
        ...,
        [148, 130,  80],
        [148, 130,  80],
        [148, 130,  80]],

       [[162, 154, 135],
        [163, 155, 136],
        [162, 154, 135],
        ...,
        [146, 128,  78],
        [147, 129,  79],
        [148, 130,  80]],

       [[164, 156, 137],
        [164, 156, 137],
        [164, 156, 137],
        ...,
        [146, 128,  78],
        [147, 129,  79],
        [147, 129,  79]]